In [9]:
import numpy as np
import scipy.linalg
import scipy.io
from scipy.sparse import csr_matrix, issparse
import matplotlib.pyplot as plt
import warnings
import time

# -------------------- Mock Helper Functions -------------------- #

def argschk(opts, **kwargs):
    """
    Mock argument checker: updates opts with kwargs.
    """
    errmsg = ""
    wrnmsg = ""
    for key, value in kwargs.items():
        if key in opts:
            opts[key] = value
        else:
            wrnmsg += f"Unknown option '{key}' ignored. "
    return opts, errmsg, wrnmsg

def compute_rms(X, A, S, M, ndata):
    """
    Mock compute_rms: calculates RMS error.
    """
    reconstruction = A @ S
    errMx = (X - reconstruction) * M
    rms = np.sqrt(np.sum(errMx ** 2) / ndata)
    return rms, errMx

def rotate_to_pca(A, Av, S, Sv, Isv, obscombj, update_bias):
    """
    Mock rotate_to_pca: does nothing.
    """
    dMu = 0
    return dMu, A, Av, S, Sv

def PrintProgress(verbose, i, n, str_progress):
    """
    Mock PrintProgress: prints progress if verbose is 2.
    """
    if verbose == 2:
        print(f'\r{str_progress} {i}/{n}', end='')

def PrintStep(verbose, lc, Aangle):
    """
    Mock PrintStep: prints step information.
    """
    if not verbose:
        return
    iter_num = len(lc['rms']) - 1
    steptime = lc['time'][-1] - lc['time'][-2] if len(lc['time']) > 1 else 0
    print(f"Step {iter_num}: rms = {lc['rms'][-1]:.6f}", end='')
    if not np.isnan(lc['prms'][-1]):
        print(f" ({lc['prms'][-1]:.6f})", end='')
    print(f", angle = {Aangle:.2e}", end='')
    if steptime > 1:
        print(f" ({int(round(steptime))} sec)")
    else:
        print(f" ({steptime:.0e} sec)")

def DisplayProgress(dsph, lc):
    """
    Mock DisplayProgress: does nothing.
    """
    if dsph.get('display', False):
        pass

def convergence_check(opts, lc, Aangle):
    """
    Mock convergence_check: never converges.
    """
    return ""

def subspace_angle(A, Aold):
    """
    Mock subspace_angle: returns a fixed small angle.
    """
    return 1e-9  # Small angle indicating convergence

def save_parameters(filename, A, S, Mu, V, Av, Muv, Sv, Isv, Va, Vmu, lc, Ir, Ic, n1x, n2x, n1, n2):
    """
    Mock save_parameters: prints a message.
    """
    print(f"Parameters saved to {filename}")
    # For actual saving, uncomment the following line
    # scipy.io.savemat(filename, {'A': A, 'S': S, 'Mu': Mu, 'V': V, 'Av': Av, 'Muv': Muv, 'Sv': Sv, 'Isv': Isv, 'Va': Va, 'Vmu': Vmu, 'lc': lc, 'Ir': Ir, 'Ic': Ic, 'n1x': n1x, 'n2x': n2x, 'n1': n1, 'n2': n2})

# -------------------- Sample Data Initialization -------------------- #

# Create a small dataset with missing values (NaN)
X = np.array([
    [85, 90, np.nan],
    [78, np.nan, 82],
    [np.nan, 88, 91],
    [92, 85, 89]
])

ncomp = 2  # Number of principal components

# Initialize options
opts = {
    'init': 'random',
    'maxiters': 5,  # Reduced iterations for testing
    'bias': 1,
    'uniquesv': 0,
    'autosave': 600,
    'filename': 'pca_f_autosave.mat',
    'minangle': 1e-8,
    'algorithm': 'vb',
    'niter_broadprior': 2,  # Reduced for testing
    'earlystop': 0,
    'rmsstop': np.array([3, 1e-4, 1e-3]),  # After 3 iterations
    'cfstop': np.array([]),                  # No cost stop criteria
    'verbose': 2,  # Set to 2 for detailed output
    'xprobe': np.array([]),
    'rotate2pca': 1,
    'display': 0
}

# Update options with any kwargs (none in this case)
opts, wrnmsg, _ = argschk(opts)
if wrnmsg:
    warnings.warn(f"Warning: {wrnmsg}")

# Handle algorithm settings
algorithmVal = opts["algorithm"]
if algorithmVal.lower() == "ppca":
    use_prior = False
    use_postvar = False
elif algorithmVal.lower() == "map":
    use_prior = True
    use_postvar = False
elif algorithmVal.lower() == "vb":
    use_prior = True
    use_postvar = True
else:
    raise ValueError(f"Wrong value of the argument 'algorithm': {algorithmVal}")

# Handle missing data
M = ~np.isnan(X)
X = np.where(np.isnan(X), 0, X)  # Replace NaNs with 0
X[X == 0] = np.finfo(float).eps  # Replace zeros with a small number

# Compute observed data points
Nobs_i = np.sum(M, axis=1)
ndata = np.sum(Nobs_i)

# No probe data in this test
Xprobe = opts['xprobe']
Mprobe = ~np.isnan(Xprobe) if Xprobe.size > 0 else np.array([])
nprobe = np.count_nonzero(Mprobe)

if nprobe == 0:
    Xprobe = np.array([])
    opts['earlystop'] = 0

# Find non-zero indices
IX, JX = np.nonzero(X)

# Handle unique Sv (not used in this test)
if opts['uniquesv']:
    nobscomb, obscombj, Isv = miscomb(M, opts['verbose'])
else:
    nobscomb = X.shape[1]
    obscombj = {j: [j] for j in range(X.shape[1])}
    Isv = list(range(nobscomb))

# Initialize parameters
# For simplicity, initialize A randomly and S randomly
A = scipy.linalg.orth(np.random.randn(X.shape[0], ncomp))
S = np.random.randn(ncomp, X.shape[1])
Mu = np.sum(X, axis=1) / Nobs_i if opts['bias'] else np.zeros(X.shape[0])
V = 1.0  # Initial variance
Av = [np.eye(ncomp) for _ in range(X.shape[0])]
Sv = [np.eye(ncomp) for _ in range(nobscomb)]
Muv = np.ones((X.shape[0], 1)) if use_postvar else np.array([])

# Initialize Va and Vmu before the loop
if use_prior:
    Va = 1000 * np.ones(ncomp)
    Vmu = 1000
else:
    Va = np.full(ncomp, np.inf)
    Vmu = np.inf

# Subtract Mu from X
def subtract_mu(Mu, X, M, Xprobe, Mprobe, update_bias=True):
    """
    Subtracts the bias vector Mu from the data matrix X and Xprobe.
    """
    if not update_bias:
        return X, Xprobe
    if isinstance(X, np.ndarray):
        X = X - np.outer(Mu, np.ones(X.shape[1])) * M
        if Xprobe.size > 0 and Mprobe.size > 0:
            Xprobe = Xprobe - np.outer(Mu, np.ones(Xprobe.shape[1])) * Mprobe
    elif issparse(X):
        # Handle sparse matrices if needed
        pass
    return X, Xprobe

X, Xprobe = subtract_mu(Mu, X, M, Xprobe, Mprobe, opts['bias'])

# Compute initial RMS errors
rms, errMx = compute_rms(X, A, S, M, ndata)
prms, _ = compute_rms(Xprobe, A, S, Mprobe, nprobe) if Xprobe.size > 0 else (np.nan, None)

# Initialize learning curves
lc = {
    'rms': [rms],
    'prms': [prms],
    'time': [0],
    'cost': [np.nan]
}

# Initialize display structure
dsph = {'display': opts['display']}

# Print first step
PrintStep(opts['verbose'], lc, 0)  # Initial step, angle=0

# -------------------- Isolated For Loop Execution -------------------- #

# Parameters of the prior for variance parameters
hpVa = 0.001
hpVb = 0.001
hpV = 0.001

time_start = time.time()
time_autosave = time_start
tic = time.time()

for iter in range(1, opts['maxiters'] + 1):
    # Update Va and Vmu if using prior and past the broad prior iterations
    if use_prior and iter > opts['niter_broadprior']:
        if opts['bias']:
            Vmu = np.sum(Mu ** 2)
            if Muv.size > 0:
                Vmu += np.sum(Muv)
            Vmu = (Vmu + 2 * hpVa) / (X.shape[0] + 2 * hpVb)
        
        Va = np.sum(A ** 2, axis=0)
        if Av:
            for i in range(X.shape[0]):
                Va += np.diag(Av[i])
        Va = (Va + 2 * hpVa) / (X.shape[0] + 2 * hpVb)
    
    # Update bias if enabled
    if opts['bias']:
        dMu = np.sum(errMx, axis=1) / Nobs_i
        if Muv.size > 0:
            Muv = V / (Nobs_i + V / Vmu)
        th = 1 / (1 + V / (Nobs_i * Vmu))
        Mu_old = Mu.copy()
        Mu = th * (Mu + dMu)
        dMu = Mu - Mu_old
        X, Xprobe = subtract_mu(dMu, X, M, Xprobe, Mprobe, True)
    
    # Update S
    if not Isv:
        for j in range(X.shape[1]):
            A_j = (M[:, j].reshape(-1, 1) * A)  # Equivalent to repmat(M[:,j],1,ncomp) .* A
            Psi = A_j.T @ A_j + V * np.eye(ncomp)  # Corrected line
            if Av:
                for i in np.where(M[:, j])[0]:
                    Psi += Av[i]
            invPsi = np.linalg.inv(Psi)
            S[:, j] = invPsi @ A_j.T @ X[:, j]
            Sv[j] = V * invPsi
            PrintProgress(opts['verbose'], j + 1, X.shape[1], 'Updating S:')
    else:
        for k in range(nobscomb):
            j = obscombj[k][0]
            A_j = (M[:, j].reshape(-1, 1) * A)
            Psi = A_j.T @ A_j + V * np.eye(ncomp)  # Corrected line
            if Av:
                for i in np.where(M[:, j])[0]:
                    Psi += Av[i]
            invPsi = np.linalg.inv(Psi)
            Sv[k] = V * invPsi
            tmp = invPsi @ A_j.T
            for j_idx in obscombj[k]:
                S[:, j_idx] = tmp @ X[:, j_idx]
            PrintProgress(opts['verbose'], k + 1, nobscomb, 'Updating S:')
    
    if opts['verbose'] == 2:
        print('\r', end='')
    
    # Rotate to PCA if enabled
    if opts['rotate2pca']:
        dMu, A, Av, S, Sv = rotate_to_pca(A, Av, S, Sv, Isv, obscombj, opts['bias'])
        if opts['bias']:
            X, Xprobe = subtract_mu(dMu, X, M, Xprobe, Mprobe, True)
            Mu += dMu
    
    # Update A
    if opts['verbose'] == 2:
        print('                                              \r', end='')
    for i in range(X.shape[0]):
        S_i = M[i, :] * S  # Corrected line; Shape: (ncomp, n2)
        Phi = S_i @ S_i.T + np.diag(V / Va)
        if Isv:
            for j in np.where(M[i, :])[0]:
                Phi += Sv[Isv[j]]
        else:
            for j in np.where(M[i, :])[0]:
                Phi += Sv[j]
        invPhi = np.linalg.inv(Phi)
        # Corrected line: transpose S_i before multiplication
        A[i, :] = (X[i, :] @ S_i.T) @ invPhi
        if Av:
            Av[i] = V * invPhi
        PrintProgress(opts['verbose'], i + 1, X.shape[0], 'Updating A:')
    if opts['verbose'] == 2:
        print('\r', end='')
    
    # Compute RMS errors
    rms, errMx = compute_rms(X, A, S, M, ndata)
    prms, _ = compute_rms(Xprobe, A, S, Mprobe, nprobe) if Xprobe.size > 0 else (np.nan, None)
    
    # Update learning curves
    lc['rms'].append(rms)
    lc['prms'].append(prms)
    lc['time'].append(time.time() - tic)
    
    # Update V
    sXv = 0
    if not Isv:
        for r in range(ndata):
            i = IX[r]
            j = JX[r]
            sXv += A[i, :] @ Sv[j] @ A[i, :].T
            if Av:
                sXv += S[:, j].T @ Av[i] @ S[:, j] + np.sum(Sv[j] * Av[i])
    else:
        for r in range(ndata):
            i = IX[r]
            j = JX[r]
            sXv += A[i, :] @ Sv[Isv[j]] @ A[i, :].T
            if Av:
                sXv += S[:, j].T @ Av[i] @ S[:, j] + np.sum(Sv[Isv[j]] * Av[i])
    
    if Muv.size > 0:
        sXv += np.sum(Muv[IX])
    
    sXv += (rms ** 2) * ndata
    V = (sXv + 2 * hpV) / (ndata + 2 * hpV)
    
    # Update cost function if applicable
    if opts['cfstop'].size > 0:
        cost = convergence_check(opts, lc, 0)  # Mock cost function
        lc['cost'].append(cost)
    
    # Display progress
    DisplayProgress(dsph, lc)
    angleA = subspace_angle(A, A)  # Using A for both to simulate zero angle
    PrintStep(opts['verbose'], lc, angleA)
    
    # Check for convergence
    convmsg = convergence_check(opts, lc, angleA)
    if convmsg:
        if use_prior and iter <= opts['niter_broadprior']:
            pass  # Do nothing
        elif opts['verbose']:
            print(convmsg)
        break
    Aold = A.copy()
    
    # Autosave if necessary (not triggered in this small test)
    current_time = time.time()
    if (current_time - time_autosave) > opts['autosave']:
        time_autosave = current_time
        if opts['verbose'] == 2:
            print('Saving ... ', end='')
        save_parameters(opts['filename'], A, S, Mu, V, Av, Muv, Sv, Isv, Va, Vmu, lc, [], [], X.shape[0], X.shape[1], X.shape[0], X.shape[1])
        if opts['verbose'] == 2:
            print('done')

# -------------------- End of For Loop -------------------- #

# Display final results
print("\nFinal Principal Components (A):")
print(A)
print("\nFinal Scores (S):")
print(S)
print("\nFinal Mean (Mu):")
print(Mu)
print("\nFinal Variance (V):")
print(V)


Step 0: rms = 2.552588, angle = 0.00e+00 (0e+00 sec)
Step 1: rms = 2.126521, angle = 1.00e-09 (3e-03 sec)
Step 2: rms = 1.737241, angle = 1.00e-09 (1e-03 sec)
Step 3: rms = 2.224062, angle = 1.00e-09 (6e-04 sec)
Step 4: rms = 2.339368, angle = 1.00e-09 (3e-04 sec)
Step 5: rms = 2.351067, angle = 1.00e-09 (3e-04 sec)

Final Principal Components (A):
[[ 0.00430341  0.02138285]
 [ 0.001349    0.00753359]
 [-0.00139448 -0.00631615]
 [-0.00556229 -0.02743181]]

Final Scores (S):
[[-0.00621679  0.00682487 -0.00072384]
 [-0.02938476  0.03046181 -0.0013611 ]]

Final Mean (Mu):
[87.37899203 79.89281906 89.37950749 88.58538272]

Final Variance (V):
21.83881499477957
